In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Cargar datos para crear el catálogo de canciones
path_train = "/content/drive/MyDrive/RUTA_AQUI/talkplay_datasets_divididos/train"
df_train = pd.read_parquet(path_train)

# Filtrar solo donde hay una recomendación real
canciones_recomendadas = df_train[df_train['is_recommendation_target'] == 1]['content_text'].unique()
song_to_idx = {song_id: i for i, song_id in enumerate(canciones_recomendadas)}
num_classes_songs = len(song_to_idx)

print(f"Diccionario creado: {num_classes_songs} canciones únicas para recomendar.")

Diccionario creado: 5791 canciones únicas para recomendar.


In [2]:
class TalkPlayDataset(Dataset):
    def __init__(self, parquet_path, song_map, tokenizer, max_len=512):
        self.df = pd.read_parquet(parquet_path)
        self.song_map = song_map
        self.tokenizer = tokenizer
        self.max_len = max_len

        #Puntos donde se hace una recomendación
        self.target_indices = self.df[self.df['is_recommendation_target'] == 1].index.tolist()

    def __len__(self):
        return len(self.target_indices)

    def __getitem__(self, idx):
        # Obtener la fila de la recomendación
        target_idx = self.target_indices[idx]
        row = self.df.loc[target_idx]
        session_id = row['session_id']
        turn_n = row['turn_number']

        # Recuperar el historial de esa sesión hasta ese turno
        historial = self.df[(self.df['session_id'] == session_id) & (self.df['turn_number'] < turn_n)]
        historial = historial.sort_values('turn_number')

        # Construir el string de entrada libre
        input_text = f"[GOAL] {row['goal_description']} [CHAT] "
        for _, h_row in historial.tail(5).iterrows(): # Tomamos los últimos 5 turnos
            speaker = "[USER]" if h_row['speaker_role'] == 'user' else "[ASST]"
            input_text += f"{speaker} {h_row['content_text']} "

        # Tokenización
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        # ID de la canción (etiqueta)
        label = self.song_map.get(row['content_text'], 0) # 0 si no existe

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [3]:
class TalkPlayIntentTransformer(nn.Module):
    def __init__(self, vocab_size, num_songs, d_model=768, nhead=12, num_layers=6):
        super(TalkPlayIntentTransformer, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Parameter(torch.zeros(1, 512, d_model))

        # Bloque Encoder
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)

        # Cabeza de Recomendación Directa
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, num_songs) # Predice la probabilidad de cada canción
        )

    def forward(self, input_ids, mask):
        # Embeddings + Posición
        x = self.embedding(input_ids) + self.pos_encoder[:, :input_ids.size(1), :]

        # Transformer
        output = self.transformer_encoder(x, src_key_padding_mask=~mask.bool())

        # Pooling (usamos el primer token para representar toda la intención)
        intent_vector = output[:, 0, :]

        # Logits de recomendación
        logits = self.classifier(intent_vector)
        return logits

In [ ]:
# Inicialización
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # O el que prefieras
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = TalkPlayDataset(path_train, song_to_idx, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

model = TalkPlayIntentTransformer(tokenizer.vocab_size, num_classes_songs).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# Loop básico
model.train()
for epoch in range(5): # Ajustar según necesidad
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} completada. Loss: {loss.item():.4f}")

In [ ]:
import os

# Definimos la ruta dentro de tu Drive
MODEL_SAVE_PATH = "/content/drive/MyDrive/RUTA_AQUI/modelos_entrenados/talkplay_transformer"

if not os.path.exists(MODEL_SAVE_PATH):
    os.makedirs(MODEL_SAVE_PATH)
    print(f"✅ Carpeta creada en: {MODEL_SAVE_PATH}")

In [ ]:
# 1. Guardar los pesos del modelo
torch.save(model.state_dict(), f"{MODEL_SAVE_PATH}/talkplay_intent_v1.pth")

# 2. Guardar el diccionario de canciones (Mapeo ID -> Index)
import json
with open(f"{MODEL_SAVE_PATH}/song_to_idx.json", "w") as f:
    json.dump(song_to_idx, f)

# 3. Guardar la configuración técnica (opcional pero recomendado)
config = {
    "vocab_size": tokenizer.vocab_size,
    "num_songs": num_classes_songs,
    "d_model": 768,
    "nhead": 12
}
with open(f"{MODEL_SAVE_PATH}/config.json", "w") as f:
    json.dump(config, f)

print(f"🚀 Todo guardado con éxito en {MODEL_SAVE_PATH}")